# PID Control — Swing · Laser · Heavy Math

**Coverage:**  
§1 PID transfer function — SymPy Laplace domain  
§2 Pendulum (swing) — linearization, PID stabilization  
§3 Root locus & Bode — SymPy + NumPy  
§4 Ziegler-Nichols tuning rules — loop over gain  
§5 Laser rate equations — PID power lock  
§6 Chinese Remainder Theorem — sampling / aliasing connection  
§7 Torch batch: PID simulation over (Kp, Ki, Kd) grid

In [ ]:
import sympy as sp
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import display, Math

sp.init_printing(use_latex='mathjax')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 110})
print('ready')

---
# §1 — PID Transfer Function (Laplace Domain)

**Plain English:** a PID controller outputs a correction signal proportional to the error $e(t)$,  
its integral (accumulated past error), and its derivative (rate of change).  
In the Laplace domain this is just multiplication by $C(s)$:

$$C(s) = K_p + \frac{K_i}{s} + K_d s = \frac{K_d s^2 + K_p s + K_i}{s}$$

The **closed-loop** transfer function with plant $G(s)$ is:
$$T(s) = \frac{C(s)G(s)}{1 + C(s)G(s)}$$

In [ ]:
s, t, Kp, Ki, Kd, wn, zeta = sp.symbols(
    's t K_p K_i K_d omega_n zeta', positive=True
)

# ── PID controller ─────────────────────────────────────────────────────
C_pid = Kp + Ki/s + Kd*s
C_pid_combined = sp.together(C_pid)
display(Math(rf'C(s) = {sp.latex(C_pid_combined)}'))

# ── loop over controller variants ─────────────────────────────────────
controllers = [
    ('P',   Kp,                  'proportional only'),
    ('PI',  Kp + Ki/s,           'eliminates steady-state error'),
    ('PD',  Kp + Kd*s,           'improves damping, sensitive to noise'),
    ('PID', Kp + Ki/s + Kd*s,    'full controller'),
]

# second-order plant: G(s) = ωn² / (s² + 2ζωns + ωn²)
G_plant = wn**2 / (s**2 + 2*zeta*wn*s + wn**2)
display(Math(rf'G_{{\text{{plant}}}}(s) = {sp.latex(G_plant)}  \quad\text{{(2nd order)}}'))
print()

print('Closed-loop T(s) = CG/(1+CG) for each controller:\n')
closed_loops = {}
for name, C, desc in controllers:
    CG = sp.simplify(C * G_plant)
    T  = sp.simplify(CG / (1 + CG))
    closed_loops[name] = T
    display(Math(rf'\text{{{name}}} \;({desc}): \quad T(s) = {sp.latex(T)}'))
    print()

# ── steady-state error: Final Value Theorem  lim_{t→∞} y(t) = lim_{s→0} s·T(s)·R(s) ──
print('Steady-state error for unit step R(s)=1/s:\n')
for name, C, desc in controllers:
    T = closed_loops[name]
    # output Y(s) = T(s)·(1/s), steady state = lim s→0 of Y(s)·s = lim T(s)
    ss = sp.limit(T, s, 0)
    err = sp.simplify(1 - ss)
    display(Math(rf'\text{{{name}}}: \quad e_{{ss}} = 1 - T(0) = {sp.latex(err)}'))

---
# §2 — The Swing (Pendulum): Linearization + PID Stabilization

**Plain English:** a pendulum is the canonical control plant.  
The nonlinear equation of motion is $ml^2\ddot{\theta} = -mgl\sin\theta - b\dot{\theta} + \tau$  
Linearize around $\theta=0$: $\sin\theta\approx\theta$ → transfer function in $s$.  
Then close the loop with PID and watch the pole locations move.

In [ ]:
theta, theta_dot, tau_s = sp.Function('theta'), sp.Function('theta_dot'), sp.Symbol('tau')
l_s, m_s, g_s, b_s = sp.symbols('l m g b', positive=True)
s2 = sp.Symbol('s')

# ── nonlinear EOM ──────────────────────────────────────────────────────
display(Math(
    r'\text{Nonlinear: }\; ml^2\ddot{\theta} + b\dot{\theta} + mgl\sin\theta = \tau'
))

# ── linearized: sin θ ≈ θ  → Laplace transform ───────────────────────
# ml²s²Θ + bsΘ + mglΘ = τ(s)
# G(s) = Θ(s)/τ(s) = 1 / (ml²s² + bs + mgl)
G_pend = 1 / (m_s*l_s**2*s2**2 + b_s*s2 + m_s*g_s*l_s)
display(Math(rf'\text{{Linearized plant: }} G(s) = \frac{{\Theta(s)}}{{\tau(s)}} = {sp.latex(G_pend)}'))

# natural frequency and damping ratio
wn_pend  = sp.sqrt(g_s / l_s)
zeta_pend = b_s / (2 * sp.sqrt(m_s**2 * g_s * l_s))
display(Math(rf'\omega_n = \sqrt{{g/l}}, \quad \zeta = b/(2ml\omega_n)'))

# ── numerical simulation: step response with PID ──────────────────────
# physical params
m_v=1.0; l_v=1.0; g_v=9.81; b_v=0.1

# loop over PID gain sets
pid_sets = [
    ('P only',      20.0,  0.0,  0.0),
    ('PD',          20.0,  0.0,  2.0),
    ('PI',          20.0,  5.0,  0.0),
    ('PID tuned',   40.0, 10.0,  5.0),
]

t_sim = np.linspace(0, 5, 2000)
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

for ax, (name, Kp_v, Ki_v, Kd_v) in zip(axes.flat, pid_sets):
    # open-loop plant numerator/denominator (scipy convention)
    plant_num = [1.0]
    plant_den = [m_v*l_v**2, b_v, m_v*g_v*l_v]

    # PID: C(s) = (Kd s² + Kp s + Ki) / s
    # Use filtered derivative: C(s) = Kp + Ki/s + Kd*s/(τ_f*s+1)  with τ_f=0.01
    tf_filt = 0.01
    pid_num = [Kd_v + Kp_v*tf_filt, Kp_v + Ki_v*tf_filt, Ki_v]
    pid_den = [tf_filt, 1.0, 0.0]

    # series: C(s)·G(s)
    ol_num = np.polymul(pid_num, plant_num)
    ol_den = np.polymul(pid_den, plant_den)

    # closed loop: T = CG/(1+CG)
    cl_num = ol_num
    cl_den = np.polyadd(ol_den, ol_num)

    sys_cl = signal.TransferFunction(cl_num, cl_den)
    _, y   = signal.step(sys_cl, T=t_sim)

    # metrics
    y_ss   = y[-1] if len(y) else 1.0
    OS     = (y.max() - y_ss) / y_ss * 100 if y_ss != 0 else 0
    idx_settle = np.where(np.abs(y - y_ss) > 0.02*abs(y_ss))[0]
    t_settle   = t_sim[idx_settle[-1]] if len(idx_settle) else 0

    ax.plot(t_sim, y, color='royalblue', lw=2)
    ax.axhline(1.0, color='k', lw=0.8, ls='--', alpha=0.5)
    ax.axhline(1.02, color='tomato', lw=0.5, ls=':', alpha=0.7)
    ax.axhline(0.98, color='tomato', lw=0.5, ls=':', alpha=0.7)
    ax.set_xlabel('time (s)'); ax.set_ylabel('θ (rad)')
    ax.set_title(f'{name}: Kp={Kp_v} Ki={Ki_v} Kd={Kd_v}\nOS={OS:.1f}%  t_settle={t_settle:.2f}s')
    ax.set_ylim(-0.2, 1.8)

plt.suptitle('Pendulum (swing) step response with PID variants', fontsize=13)
plt.tight_layout(); plt.show()

---
# §3 — Root Locus and Bode Plot

**Plain English:**  
- **Root locus**: as you sweep gain $K$, where do the closed-loop poles go? Stable = left half-plane.  
- **Bode plot**: gain and phase of the open-loop $C(j\omega)G(j\omega)$ vs frequency.  
  - **Gain margin**: how much you can increase gain before instability  
  - **Phase margin**: how much phase lag before instability (want > 45°)

In [ ]:
# ── pendulum open-loop with PID ────────────────────────────────────────
Kp_v=40; Ki_v=10; Kd_v=5; tf_filt=0.01
pid_num = [Kd_v + Kp_v*tf_filt, Kp_v + Ki_v*tf_filt, Ki_v]
pid_den = [tf_filt, 1.0, 0.0]
pl_num  = [1.0]
pl_den  = [m_v*l_v**2, b_v, m_v*g_v*l_v]
ol_num  = np.polymul(pid_num, pl_num)
ol_den  = np.polymul(pid_den, pl_den)
sys_ol  = signal.TransferFunction(ol_num, ol_den)

# ── Bode plot ─────────────────────────────────────────────────────────
w, H = signal.freqs(ol_num, ol_den, worN=np.logspace(-1, 3, 2000))
mag_db = 20*np.log10(np.abs(H) + 1e-20)
phase_deg = np.angle(H, deg=True)

# gain margin: frequency where phase = -180°
idx_pm180 = np.argmin(np.abs(phase_deg + 180))
GM = -mag_db[idx_pm180]
# phase margin: frequency where |H| = 1 (0 dB)
idx_0dB = np.argmin(np.abs(mag_db))
PM = phase_deg[idx_0dB] + 180

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].semilogx(w, mag_db, color='royalblue', lw=2)
axes[0].axhline(0, color='k', lw=0.8, ls='--')
axes[0].axvline(w[idx_0dB], color='tomato', lw=1, ls='--', label=f'0 dB @ ω={w[idx_0dB]:.2f}')
axes[0].set_ylabel('Magnitude (dB)'); axes[0].set_title('Bode Plot — PID × Pendulum')
axes[0].legend(); axes[0].grid(True, which='both', alpha=0.3)

axes[1].semilogx(w, phase_deg, color='seagreen', lw=2)
axes[1].axhline(-180, color='k', lw=0.8, ls='--')
axes[1].axvline(w[idx_pm180], color='tomato', lw=1, ls='--', label=f'-180° @ ω={w[idx_pm180]:.2f}')
axes[1].set_ylabel('Phase (deg)'); axes[1].set_xlabel('ω (rad/s)')
axes[1].legend(); axes[1].grid(True, which='both', alpha=0.3)

plt.suptitle(f'Gain margin = {GM:.1f} dB  |  Phase margin = {PM:.1f}°', fontsize=12)
plt.tight_layout(); plt.show()

# ── Root locus: loop over gain K ──────────────────────────────────────
K_vals = np.linspace(0.1, 200, 5000)
poles_all = []
for K in K_vals:
    cl_d = np.polyadd(ol_den, K * ol_num)
    poles_all.append(np.roots(cl_d))

poles_arr = np.array(poles_all)   # (N_K, n_poles)

fig, ax = plt.subplots(figsize=(8, 6))
colors_rl = plt.cm.viridis(np.linspace(0, 1, poles_arr.shape[1]))
for pi in range(poles_arr.shape[1]):
    sc = ax.scatter(poles_arr[:, pi].real, poles_arr[:, pi].imag,
                    c=K_vals, cmap='viridis', s=2, alpha=0.6)
ax.axvline(0, color='k', lw=1)
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Re(s)'); ax.set_ylabel('Im(s)')
ax.set_title('Root locus — poles vs gain K (PID × pendulum)')
ax.set_xlim(-80, 10)
plt.colorbar(sc, ax=ax, label='K'); plt.tight_layout(); plt.show()

---
# §4 — Ziegler-Nichols Tuning Rules (Loop over Methods)

**Plain English:** Ziegler-Nichols (1942) gives closed-form formulas for $K_p, K_i, K_d$  
from two measured parameters: **ultimate gain** $K_u$ and **ultimate period** $T_u$.  
Loop over all five Z-N controller types and compute the gain set.

In [ ]:
Ku_s, Tu_s = sp.symbols('K_u T_u', positive=True)

# Ziegler-Nichols table (classical)
zn_rules = [
    ('P',       Ku_s/2,              0,                   0),
    ('PI',      Ku_s*sp.Rational(9,20), Ku_s*sp.Rational(9,20)*sp.Rational(2,1)/Tu_s, 0),
    ('PD',      Ku_s*sp.Rational(4,5),  0,                Ku_s*sp.Rational(4,5)*Tu_s/8),
    ('PID',     Ku_s/sp.Integer(2)*sp.Rational(3,5)*2,
                Ku_s*sp.Rational(3,5)*2/(Tu_s*sp.Rational(1,2)),
                Ku_s*sp.Rational(3,5)*2*Tu_s/8),
    ('PID (some overshoot)', Ku_s/3, 2*Ku_s/(3*Tu_s), Ku_s*Tu_s/9),
]

# Standard textbook table
zn_standard = [
    ('P',                    Ku_s/2,                0,                        0),
    ('PI',                   Ku_s*sp.Rational(9,20), Ku_s/(sp.Rational(9,20)*Tu_s/sp.Rational(10,12)), 0),
    ('PID (classic)',        Ku_s*sp.Rational(3,5),  2*Ku_s*sp.Rational(3,5)/Tu_s, Ku_s*sp.Rational(3,5)*Tu_s/8),
    ('PID (no overshoot)',   Ku_s/5,                 2*Ku_s/(5*Tu_s),          Ku_s*Tu_s/15),
    ('PID (some overshoot)', Ku_s/3,                 2*Ku_s/(3*Tu_s),          Ku_s*Tu_s/9),
]

print('Ziegler-Nichols tuning rules:\n')
print(f'{"Method":25}  {"Kp":30}  {"Ki":30}  {"Kd"}')
print('-'*110)
for name, kp, ki, kd in zn_standard:
    display(Math(
        rf'\text{{{name}}}: \quad K_p={sp.latex(kp)},\; K_i={sp.latex(ki)},\; K_d={sp.latex(kd)}'
    ))

# ── NumPy: simulate step response for each Z-N set ────────────────────
# Find Ku numerically: increase K until sustained oscillation
# For pendulum: open loop with pure P → find K where poles hit jω axis
# closed: s(ml²s² + bs + mgl) + K = 0  →  ml²s³ + bs² + mgls + K = 0
# Routh: unstable when b*mgl = ml²*K  →  Ku = b*mgl/(ml²) * ... 
# More precisely: for 3rd order ml²s³+bs²+mgls+K, Routh condition: b*mgl > ml²*K → Ku = b*g/l
Ku_v = b_v * g_v / l_v   # Routh-Hurwitz boundary for pure P on pendulum
# sustained oscillation frequency: ω_u = √(mgl/(ml²)) = √(g/l)
wu_v = np.sqrt(g_v / l_v)
Tu_v = 2*np.pi / wu_v

print(f'\nPendulum ultimate gain Ku = {Ku_v:.4f},  Tu = {Tu_v:.4f} s,  ωu = {wu_v:.4f} rad/s')

# substitute numerical values into Z-N rules
t_sim2 = np.linspace(0, 8, 3000)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

zn_num = [
    ('P (Z-N)',            Ku_v/2,         0,               0),
    ('PID classic (Z-N)',  0.6*Ku_v,       2*0.6*Ku_v/Tu_v, 0.6*Ku_v*Tu_v/8),
    ('PID no-OS (Z-N)',    Ku_v/5,         2*Ku_v/(5*Tu_v), Ku_v*Tu_v/15),
]
colors_zn = ['royalblue', 'seagreen', 'tomato']

for ax, (name, Kp_v, Ki_v, Kd_v), col in zip(axes, zn_num, colors_zn):
    pid_num2 = [Kd_v + Kp_v*tf_filt, Kp_v + Ki_v*tf_filt, Ki_v]
    pid_den2 = [tf_filt, 1.0, 0.0]
    cl_n2    = np.polymul(pid_num2, pl_num)
    cl_d2    = np.polyadd(np.polymul(pid_den2, pl_den), cl_n2)
    try:
        sys2 = signal.TransferFunction(cl_n2, cl_d2)
        _, y2 = signal.step(sys2, T=t_sim2)
        ax.plot(t_sim2, y2, color=col, lw=2)
        ax.axhline(1, color='k', lw=0.8, ls='--')
        OS2 = max(0, (y2.max()-1)*100)
        ax.set_title(f'{name}\nKp={Kp_v:.2f} Ki={Ki_v:.2f} Kd={Kd_v:.2f}\nOS={OS2:.1f}%')
    except Exception as ex:
        ax.set_title(f'{name}\nUnstable: {ex}')
    ax.set_xlabel('t (s)'); ax.set_ylabel('θ'); ax.set_ylim(-0.3, 2.0)

plt.suptitle('Ziegler-Nichols tuned step responses (pendulum)', fontsize=12)
plt.tight_layout(); plt.show()

---
# §5 — Laser Rate Equations + PID Power Lock

**Plain English:** a semiconductor laser is described by two coupled ODEs:  
- $\dot{N} = J/ed - N/\tau_e - v_g g(N) S$ — carrier density ($N$)  
- $\dot{S} = v_g g(N) S - S/\tau_p + \beta N/\tau_e$ — photon density ($S$)  

where $g(N) = a(N-N_0)$ is the linear gain.  
**PID power lock:** measure output power $P \propto S$, compare to setpoint, feed correction into drive current $J$.

In [ ]:
# ── laser parameters (InGaAsP, 1550 nm class) ─────────────────────────
params = dict(
    tau_e = 2e-9,     # carrier lifetime  (s)
    tau_p = 2e-12,    # photon lifetime   (s)
    vg    = 8e9,      # group velocity    (m/s)
    a_gain= 3e-20,    # differential gain (m²)
    N0    = 1.5e24,   # transparency density (m⁻³)
    beta  = 1e-5,     # spontaneous emission factor
    e_q   = 1.602e-19,
    d     = 200e-9,   # active layer thickness (m)
    V_act = 1e-16,    # active volume (m³)
)

def laser_rhs(t_val, y, J_fn, p):
    """Rate equations RHS. y = [N, S]."""
    N, S = y
    S    = max(S, 0.0)
    J    = J_fn(t_val)
    g    = p['vg'] * p['a_gain'] * max(N - p['N0'], 0)
    dN   = J / (p['e_q'] * p['d']) - N / p['tau_e'] - g * S
    dS   = g * S - S / p['tau_p'] + p['beta'] * N / p['tau_e']
    return [dN, dS]

# ── open-loop: step in current density ────────────────────────────────
from scipy.integrate import solve_ivp

J_th  = 1.5e9    # threshold current density (A/m²) — approximate
J_bias= 2.5e9    # bias point

t_span = (0, 10e-9)
t_eval = np.linspace(*t_span, 5000)
y0     = [params['N0'], 1e18]

# step from 2.5 to 3.5 × J_th at t=2 ns
def J_step(t):
    return J_bias * (1 + 0.4 * (t > 2e-9))

sol_ol = solve_ivp(laser_rhs, t_span, y0, t_eval=t_eval,
                   args=(J_step, params), method='RK45',
                   rtol=1e-6, atol=1e-12)

# ── PID power lock simulation ─────────────────────────────────────────
dt     = t_eval[1] - t_eval[0]
P_set  = sol_ol.y[1, -1] * 1.2   # setpoint = 20% above final open-loop

# PID gains (hand-tuned for ns timescale)
Kp_L = 1e-12; Ki_L = 5e-4; Kd_L = 1e-20

N_arr  = np.zeros(len(t_eval))
S_arr  = np.zeros(len(t_eval))
J_arr  = np.zeros(len(t_eval))
N_arr[0], S_arr[0] = y0
J_arr[0] = J_bias
integral_err = 0.0; prev_err = 0.0

for i in range(1, len(t_eval)):
    # PID error on photon density
    err         = P_set - S_arr[i-1]
    integral_err += err * dt
    deriv_err   = (err - prev_err) / dt if i > 1 else 0
    J_corr      = Kp_L*err + Ki_L*integral_err + Kd_L*deriv_err
    J_arr[i]    = np.clip(J_bias + J_corr, 0.5*J_bias, 3*J_bias)
    prev_err    = err

    # one RK4 step
    y_curr = [N_arr[i-1], S_arr[i-1]]
    J_now  = lambda tt: J_arr[i]
    k1 = laser_rhs(t_eval[i-1], y_curr, J_now, params)
    k2 = laser_rhs(t_eval[i-1]+dt/2, [y_curr[j]+dt/2*k1[j] for j in range(2)], J_now, params)
    k3 = laser_rhs(t_eval[i-1]+dt/2, [y_curr[j]+dt/2*k2[j] for j in range(2)], J_now, params)
    k4 = laser_rhs(t_eval[i-1]+dt,   [y_curr[j]+dt*k3[j]   for j in range(2)], J_now, params)
    for j in range(2):
        val = y_curr[j] + dt*(k1[j]+2*k2[j]+2*k3[j]+k4[j])/6
        if j==0: N_arr[i] = max(val, 0)
        else:    S_arr[i] = max(val, 0)

# ── plot ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
t_ns = t_eval * 1e9

axes[0,0].plot(t_ns, sol_ol.y[1], color='royalblue', lw=1.5, label='S (photons)')
axes[0,0].axvline(2, color='tomato', ls='--', lw=1, label='step at 2 ns')
axes[0,0].set_title('Open-loop: photon density S'); axes[0,0].set_xlabel('t (ns)')
axes[0,0].legend(); axes[0,0].set_ylabel('S (m⁻³)')

axes[0,1].plot(t_ns, sol_ol.y[0], color='seagreen', lw=1.5)
axes[0,1].axhline(params['N0'], color='k', ls='--', lw=0.8, label='N₀ (transparency)')
axes[0,1].set_title('Open-loop: carrier density N'); axes[0,1].set_xlabel('t (ns)')
axes[0,1].legend(); axes[0,1].set_ylabel('N (m⁻³)')

axes[1,0].plot(t_ns, S_arr, color='royalblue', lw=1.5, label='S (PID)')
axes[1,0].axhline(P_set, color='tomato', ls='--', lw=1.5, label=f'setpoint={P_set:.2e}')
axes[1,0].set_title('PID power lock: photon density'); axes[1,0].set_xlabel('t (ns)')
axes[1,0].legend(); axes[1,0].set_ylabel('S (m⁻³)')

axes[1,1].plot(t_ns, J_arr/J_bias, color='seagreen', lw=1.5)
axes[1,1].axhline(1.0, color='k', ls='--', lw=0.8)
axes[1,1].set_title('PID drive current (normalized)'); axes[1,1].set_xlabel('t (ns)')
axes[1,1].set_ylabel('J / J_bias')

plt.suptitle('Laser rate equations: open-loop step vs PID power lock', fontsize=12)
plt.tight_layout(); plt.show()

---
# §6 — Chinese Remainder Theorem + Sampling / Aliasing

**Plain English:** CRT says: if you know $x \bmod m_1$, $x \bmod m_2$, … (with coprime $m_i$)  
you can reconstruct $x$ exactly up to $M = \prod m_i$.  
This is *exactly* the logic behind **multi-rate sampling and aliasing resolution**:  
sample at several coprime rates $f_s^{(i)}$, then CRT reconstructs the true frequency  
even when each individual sampler aliases.  Used in RF spectral analysis and photonic ADCs.

In [ ]:
# ── SymPy CRT ──────────────────────────────────────────────────────────
def crt_sympy(remainders, moduli):
    """Chinese Remainder Theorem via sympy."""
    M = sp.Mul(*moduli)
    x = sp.Integer(0)
    for r, m in zip(remainders, moduli):
        Mi = M // m
        yi = sp.mod_inverse(Mi, m)
        x  = x + r * Mi * yi
    return sp.Mod(x, M), M

# ── loop: recover several unknown integers from their residues ─────────
moduli = [sp.Integer(m) for m in [3, 5, 7, 11, 13]]
M_total = sp.Mul(*moduli)
print(f'Moduli: {[int(m) for m in moduli]}  →  M = {M_total}\n')

test_vals = [42, 137, 1001, 3002, 15014]
print(f'{"True x":10}  {"Residues":40}  {"CRT recovered":15}  match?')
print('-'*75)
for x_true in test_vals:
    res = [sp.Integer(x_true % int(m)) for m in moduli]
    x_rec, _ = crt_sympy(res, moduli)
    match = '✓' if int(x_rec) == x_true % int(M_total) else '✗'
    print(f'{x_true:10d}  {str([int(r) for r in res]):40}  {int(x_rec):15d}  {match}')

# ── sampling / aliasing application ───────────────────────────────────
print('\nPhotonic ADC: multi-rate CRT frequency recovery')
print('True signal frequency: f = 2.3 GHz')
print()

f_true = 2.3e9   # Hz
sample_rates = [3.0e9, 5.0e9, 7.0e9]   # GHz, mutually coprime (as integers in GHz units)

# Each sampler sees f_alias = f mod f_s
aliases = [f_true % fs for fs in sample_rates]
print(f'  Sampler rates:   {[f"{fs/1e9:.0f} GHz" for fs in sample_rates]}')
print(f'  Aliased freqs:   {[f"{a/1e9:.3f} GHz" for a in aliases]}')

# CRT recovery (integer arithmetic in MHz)
scale = 1e6
mod_int   = [int(fs/scale) for fs in sample_rates]
rem_int   = [int(round(a/scale)) for a in aliases]
M_samp    = 1
for m in mod_int: M_samp *= m

x_rec_int = sp.Integer(0)
for r, m in zip(rem_int, mod_int):
    Mi = M_samp // m
    yi = sp.mod_inverse(Mi, m)
    x_rec_int += r * Mi * yi
f_rec = int(x_rec_int % M_samp) * scale
print(f'  CRT recovered:   {f_rec/1e9:.3f} GHz  (true: {f_true/1e9:.3f} GHz)')
print(f'  Unambiguous range: 0 – {M_samp/1e3:.0f} GHz')

# ── visualization: aliased signals at each sampler ────────────────────
t_v = np.linspace(0, 5e-9, 5000)
sig_true = np.cos(2*np.pi*f_true*t_v)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, fs, fa in zip(axes, sample_rates, aliases):
    t_s = np.arange(0, 5e-9, 1/fs)
    ax.plot(t_v*1e9, sig_true, color='k', lw=0.8, alpha=0.4, label='true')
    ax.plot(t_v*1e9, np.cos(2*np.pi*fa*t_v), color='royalblue', lw=1.5,
            ls='--', label=f'alias {fa/1e9:.2f} GHz')
    ax.scatter(t_s*1e9, np.cos(2*np.pi*f_true*t_s), s=20, color='tomato', zorder=5)
    ax.set_title(f'fs = {fs/1e9:.0f} GHz'); ax.set_xlabel('t (ns)')
    ax.legend(fontsize=8)

plt.suptitle(f'CRT multi-rate sampling: f={f_true/1e9} GHz aliased to ['+
             ', '.join([f'{a/1e9:.2f}' for a in aliases])+'] GHz', fontsize=11)
plt.tight_layout(); plt.show()

---
# §7 — Torch Batch: PID Simulation over (Kp, Ki, Kd) Grid

**Plain English:** instead of simulating one gain set at a time, use Torch to run  
thousands of PID configurations in parallel and find the Pareto-optimal trade-off  
between **overshoot** and **settling time** — the core of automatic PID tuning.

In [ ]:
# ── discrete-time PID simulation in Torch (vectorized over gains) ─────
# Plant: second-order  G(z) ≈ bilinear transform of G(s)=ωn²/(s²+2ζωns+ωn²)
# Simple Euler forward discretization for speed

Ts   = 0.002   # sample period (s)
N_t  = 2500    # steps = 5 s
wn_v2  = np.sqrt(g_v/l_v)
zeta_v = b_v / (2*m_v*wn_v2*l_v**2)

# plant: x[k+1] = A x[k] + B u[k],  y[k] = C x[k]
A_p = np.array([[1, Ts],
                [-wn_v2**2*Ts, 1 - 2*zeta_v*wn_v2*Ts]])
B_p = np.array([0, wn_v2**2*Ts])
C_p = np.array([1, 0])

A_t = torch.tensor(A_p, dtype=torch.float64)
B_t = torch.tensor(B_p, dtype=torch.float64)
C_t = torch.tensor(C_p, dtype=torch.float64)

# ── gain grid ─────────────────────────────────────────────────────────
Kp_grid = torch.linspace(1, 80, 20, dtype=torch.float64)
Ki_grid = torch.linspace(0, 30, 10, dtype=torch.float64)
Kd_grid = torch.linspace(0, 15, 10, dtype=torch.float64)

Kp_all, Ki_all, Kd_all = torch.meshgrid(Kp_grid, Ki_grid, Kd_grid, indexing='ij')
Kp_flat = Kp_all.reshape(-1)
Ki_flat = Ki_all.reshape(-1)
Kd_flat = Kd_all.reshape(-1)
N_batch = Kp_flat.shape[0]
print(f'Simulating {N_batch} PID configurations...')

# state: (N_batch, 2),  error integral: (N_batch,),  prev_error: (N_batch,)
x_batch   = torch.zeros(N_batch, 2, dtype=torch.float64)
int_batch = torch.zeros(N_batch, dtype=torch.float64)
ep_batch  = torch.zeros(N_batch, dtype=torch.float64)

r_ref = torch.ones(N_batch, dtype=torch.float64)   # unit step setpoint
y_hist = torch.zeros(N_batch, N_t, dtype=torch.float64)
u_hist = torch.zeros(N_batch, N_t, dtype=torch.float64)

for k in range(N_t):
    y_k       = (C_t @ x_batch.T)              # (N_batch,)
    err       = r_ref - y_k
    int_batch = int_batch + err * Ts
    derr      = (err - ep_batch) / Ts
    u_k       = Kp_flat*err + Ki_flat*int_batch + Kd_flat*derr
    u_k       = torch.clamp(u_k, -200, 200)
    # state update: x = A x + B u
    x_batch   = (A_t @ x_batch.T).T + torch.outer(u_k, B_t)
    ep_batch  = err
    y_hist[:, k] = y_k
    u_hist[:, k] = u_k

# ── compute metrics for all batch ─────────────────────────────────────
y_ss    = y_hist[:, -1]
overshoot = ((y_hist.max(dim=1).values - y_ss) / (y_ss.abs() + 1e-6) * 100).clamp(0)

# settling time: last index where |y - y_ss| > 2%
settled = (y_hist - y_ss.unsqueeze(1)).abs() > 0.02 * y_ss.abs().unsqueeze(1)
t_settle_idx = settled.float().flip(dims=[1]).argmax(dim=1)
t_settle_sec = (N_t - t_settle_idx.float()) * Ts

# only stable: |y_ss| in [0.8, 1.2]
stable_mask = (y_ss.abs() > 0.8) & (y_ss.abs() < 1.2)
OS_s    = overshoot[stable_mask].numpy()
Ts_s    = t_settle_sec[stable_mask].numpy()
Kp_s    = Kp_flat[stable_mask].numpy()

print(f'{stable_mask.sum().item()} stable configurations out of {N_batch}')

# ── Pareto front: min overshoot AND min settling time ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc = axes[0].scatter(OS_s, Ts_s, c=Kp_s, cmap='viridis', s=8, alpha=0.6)
plt.colorbar(sc, ax=axes[0], label='Kp')
axes[0].set_xlabel('Overshoot (%)')
axes[0].set_ylabel('Settling time (s)')
axes[0].set_title('PID design space: OS vs t_settle\n(Torch batch over 2000 gain configs)')
axes[0].set_xlim(0, 100); axes[0].set_ylim(0, 5)

# ── plot best 5 by lowest OS·t_settle product ─────────────────────────
score = OS_s * Ts_s
best5 = np.argsort(score)[:5]
idx_all = np.where(stable_mask.numpy())[0]

t_axis = np.arange(N_t) * Ts
colors5 = plt.cm.tab10(np.linspace(0,1,5))
for rank, (bi, col) in enumerate(zip(best5, colors5)):
    gi = idx_all[bi]
    axes[1].plot(t_axis, y_hist[gi].numpy(), color=col, lw=1.5,
                 label=f'Kp={Kp_flat[gi]:.0f} Ki={Ki_flat[gi]:.0f} Kd={Kd_flat[gi]:.0f}')
axes[1].axhline(1, color='k', lw=0.8, ls='--')
axes[1].set_xlabel('t (s)'); axes[1].set_ylabel('y(t)')
axes[1].set_title('Top 5 PID configs (min OS × t_settle)')
axes[1].legend(fontsize=8); axes[1].set_ylim(-0.1, 1.8)

plt.suptitle('Torch batch PID auto-tuning — pendulum plant', fontsize=12)
plt.tight_layout(); plt.show()